# 🚀 Prototipo ETL: Google Earth Engine → Cloudflare R2 (Formato Zarr)

Este notebook hace exactamente lo mismo que `DatasetPRO3.ipynb`, pero **sin usar `.getInfo()`**.

En cambio:
1. Descarga los datos de Sentinel-5P usando `xarray` + el motor `xee` (motor de Earth Engine para xarray).
2. Guarda el resultado como un archivo **Zarr** directamente en tu bucket de **Cloudflare R2**.
3. Lee el archivo Zarr desde R2 y genera la misma gráfica de serie de tiempo.

---
**⚠️ IMPORTANTE: Antes de ejecutar, debes:**
1. Ir a la consola de Cloudflare → R2 → **Manage R2 API Tokens** → **Create API token**.
2. Copiar el **Access Key ID**, el **Secret Access Key** y la **URL del Endpoint S3**.
3. Reemplazar los valores `'TU_ACCESS_KEY_ID'`, `'TU_SECRET_ACCESS_KEY'` y `'TU_ENDPOINT_URL'` en la celda 2.
4. Reemplazar `'TU_NOMBRE_DE_BUCKET'` por el nombre real de tu bucket de R2.

In [ ]:
# ============================================================
# CELDA 1: INSTALACIÓN DE LIBRERÍAS
# ============================================================
# xarray: Maneja matrices multidimensionales (cubos de datos)
# zarr: Formato de almacenamiento eficiente para datos científicos
# s3fs: Conecta Python con cualquier almacenamiento compatible S3 (como R2)
# xee: Motor que conecta Google Earth Engine con xarray directamente
# geemap: Herramientas auxiliares de Earth Engine

!pip install xarray zarr s3fs xee geemap -q

print('✅ Librerías instaladas correctamente.')

In [ ]:
# ============================================================
# CELDA 2: AUTENTICACIÓN Y CONFIGURACIÓN
# ============================================================
import ee
import xarray as xr
import s3fs
import zarr
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- 2a. Autenticación con Google Earth Engine ---
ee.Authenticate()
ee.Initialize(project='proyecto3ia-494900')
print('✅ Google Earth Engine autenticado.')

# --- 2b. Configuración de conexión a Cloudflare R2 ---
# ⚠️ REEMPLAZA ESTOS VALORES con los que obtengas de la consola de Cloudflare R2
R2_ACCESS_KEY_ID     = 'TU_ACCESS_KEY_ID'        # Ejemplo: 'a1b2c3d4e5f6...'
R2_SECRET_ACCESS_KEY = 'TU_SECRET_ACCESS_KEY'     # Ejemplo: 'x9y8z7w6v5...'
R2_ENDPOINT_URL      = 'TU_ENDPOINT_URL'          # Ejemplo: 'https://<account_id>.r2.cloudflarestorage.com'
R2_BUCKET_NAME       = 'TU_NOMBRE_DE_BUCKET'      # Ejemplo: 'geovision-cali'

# Crear el objeto de conexión S3 hacia Cloudflare R2
s3 = s3fs.S3FileSystem(
    key=R2_ACCESS_KEY_ID,
    secret=R2_SECRET_ACCESS_KEY,
    endpoint_url=R2_ENDPOINT_URL,
    client_kwargs={'region_name': 'auto'}  # R2 usa 'auto' como región
)

print(f'✅ Conexión a Cloudflare R2 configurada. Bucket: {R2_BUCKET_NAME}')

In [ ]:
# ============================================================
# CELDA 3: EXTRACCIÓN DE DATOS DESDE GOOGLE EARTH ENGINE
# ============================================================
# En vez de usar .getRegion().getInfo() (que explota con datos grandes),
# usamos xarray con el motor 'xee' para mantener la estructura matemática.

import xee  # Importar para registrar el motor 'ee' en xarray

# 1. Definir el área de estudio: Bounding Box de Cali
#    (Igual que en tu notebook original)
cali_roi = ee.Geometry.BBox(-76.60, 3.30, -76.40, 3.55)

# 2. Definir el rango de fechas (muestra pequeña: 3 meses)
FECHA_INICIO = '2024-01-01'
FECHA_FIN    = '2024-03-31'

# 3. Filtrar la colección de Sentinel-5P (NO2)
s5p_collection = (
    ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
    .filterBounds(cali_roi)
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .select('tropospheric_NO2_column_number_density')
)

print(f'Colección filtrada. Descargando datos como xarray Dataset...')
print(f'Esto puede tardar unos 30-60 segundos dependiendo de la conexión.')

# 4. Convertir la ImageCollection de GEE a un Dataset de xarray
#    El motor 'ee' (proporcionado por la librería xee) se encarga de
#    descargar los datos por pedazos (chunks) sin explotar la RAM.
#    scale=7000 → resolución de ~7km (Sentinel-5P tiene ~3.5x5.5 km nativo)
#    geometry=cali_roi → solo descarga los píxeles dentro del bounding box de Cali
ds = xr.open_dataset(
    s5p_collection,
    engine='ee',
    scale=7000,         # Resolución en metros
    geometry=cali_roi,  # Solo los píxeles de Cali
    crs='EPSG:4326'     # Sistema de coordenadas geográficas estándar
)

print(f'\n✅ ¡Datos descargados exitosamente!')
print(f'\n--- Estructura del Dataset ---')
print(ds)
print(f'\n--- Dimensiones ---')
print(f'  Tiempo:   {ds.dims["time"]} pasos temporales')
print(f'  Latitud:  {ds.dims["lat"]} píxeles')
print(f'  Longitud: {ds.dims["lon"]} píxeles')

In [ ]:
# ============================================================
# CELDA 4: GUARDAR EN CLOUDFLARE R2 COMO ZARR
# ============================================================
# Este es el paso clave del ETL:
# Los datos pasan de Google Earth Engine → directamente a tu nube R2.
# NO se guardan en tu disco duro local (excepto temporalmente en RAM).

# 1. Primero, computamos (materializamos) los datos lazy de xarray
#    Esto fuerza la descarga real de los píxeles desde GEE.
print('Materializando datos (descargando píxeles reales desde GEE)...')
ds_computed = ds.compute()
print(f'✅ Datos materializados. Tamaño en memoria: {ds_computed.nbytes / 1024:.1f} KB')

# 2. Definir la ruta en R2 donde guardaremos el Zarr
zarr_path = f'{R2_BUCKET_NAME}/muestra_cali/sentinel_5p/NO2'
print(f'\nGuardando en: s3://{zarr_path}')

# 3. Crear un "store" (almacén) que apunta a R2
store = s3fs.S3Map(root=zarr_path, s3=s3, check=False)

# 4. Guardar el Dataset como Zarr en R2
ds_computed.to_zarr(store=store, mode='w', consolidated=True)

print(f'\n✅ ¡Datos guardados exitosamente en Cloudflare R2!')
print(f'   Ruta: s3://{zarr_path}')
print(f'   Formato: Zarr (chunks comprimidos)')

In [ ]:
# ============================================================
# CELDA 5: VERIFICAR QUE EL ARCHIVO EXISTE EN R2
# ============================================================
# Listamos los archivos dentro de la carpeta Zarr para confirmar
# que todo se guardó correctamente.

archivos_en_r2 = s3.ls(zarr_path)
print(f'📂 Contenido de s3://{zarr_path}:')
for archivo in archivos_en_r2:
    print(f'   └── {archivo.split("/")[-1]}')

print(f'\n✅ Se encontraron {len(archivos_en_r2)} elementos en la carpeta Zarr.')

In [ ]:
# ============================================================
# CELDA 6: LEER DATOS DESDE R2 Y HACER EL ANÁLISIS (EDA)
# ============================================================
# Ahora simulamos lo que haría un científico de datos:
# NO tiene los datos locales, los lee directamente desde la nube.

print('📡 Conectando a Cloudflare R2 para leer el Zarr...')

# 1. Abrir el store de lectura
store_lectura = s3fs.S3Map(root=zarr_path, s3=s3, check=False)

# 2. Abrir el Dataset desde R2 (lectura lazy: no carga todo en RAM)
ds_from_r2 = xr.open_zarr(store=store_lectura, consolidated=True)

print(f'✅ Dataset leído desde R2.')
print(ds_from_r2)

# 3. Seleccionar el punto central de Cali (igual que en tu notebook original)
#    method='nearest' busca el píxel más cercano a esas coordenadas
PUNTO_LAT = 3.45
PUNTO_LON = -76.53

serie_no2 = ds_from_r2['tropospheric_NO2_column_number_density'].sel(
    lat=PUNTO_LAT,
    lon=PUNTO_LON,
    method='nearest'
)

# 4. Convertir a DataFrame de Pandas (solo el punto, no toda la cuadrícula)
df = serie_no2.to_dataframe().reset_index()
df = df.rename(columns={'tropospheric_NO2_column_number_density': 'NO2'})
df = df.dropna(subset=['NO2'])  # Quitar días sin datos (nubes, etc.)
df = df.rename(columns={'time': 'fecha'})

print(f'\n✅ Se extrajeron {len(df)} días de datos válidos desde R2.')
print(f'\nAsí se ven los datos leídos desde la nube:')
display(df[['fecha', 'NO2']].head(10))

In [ ]:
# ============================================================
# CELDA 7: GRÁFICA DE SERIE DE TIEMPO (IGUAL QUE EL ORIGINAL)
# ============================================================
# Esta gráfica es idéntica a la de tu notebook DatasetPRO3.ipynb,
# pero los datos vienen de R2 (Zarr) en vez de .getInfo().

plt.figure(figsize=(12, 5))
plt.plot(df['fecha'], df['NO2'], marker='o', linestyle='-', color='firebrick', markersize=4)

# Línea de tendencia simple (Promedio del trimestre)
promedio = df['NO2'].mean()
plt.axhline(y=promedio, color='blue', linestyle='--', label=f'Promedio ({promedio:.6f})')

plt.title('Serie de Tiempo: Contaminación por NO₂ en el Centro de Cali (Ene-Mar 2024)\n'
          '📡 Datos leídos desde Cloudflare R2 (formato Zarr)', fontsize=14)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Concentración de NO₂ (mol/m²)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Estadísticas básicas
print('\n--- Estadísticas Descriptivas ---')
print(f'  Mínimo:  {df["NO2"].min():.8f} mol/m²')
print(f'  Máximo:  {df["NO2"].max():.8f} mol/m²')
print(f'  Media:   {df["NO2"].mean():.8f} mol/m²')
print(f'  Mediana: {df["NO2"].median():.8f} mol/m²')
print(f'  Std:     {df["NO2"].std():.8f} mol/m²')

---
## ✅ Resumen del Flujo ETL Completo

| Paso | Descripción | Herramienta |
|------|-------------|-------------|
| **1. Extract** | Descarga de Sentinel-5P desde Google Earth Engine | `xarray` + motor `xee` |
| **2. Transform** | Recorte geográfico (Bounding Box de Cali) + Filtro temporal | GEE Server-side |
| **3. Load** | Guardado como Zarr en Cloudflare R2 | `s3fs` + `.to_zarr()` |
| **4. Analyze** | Lectura desde R2 + Serie de Tiempo | `xr.open_zarr()` + Matplotlib |

### 🔑 Diferencia clave con el notebook original:
- **Antes:** `.getRegion().getInfo()` → Descargaba TODO a la RAM como texto plano. Explota con datos grandes.
- **Ahora:** `xarray` + `xee` → Descarga por chunks (pedazos), mantiene la estructura matemática, y escala a 50 GB+.